# [Step 3] 데이터 EDA 및 시각화

In [16]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
BASE_DIR = Path(".")
DATA_RAW = BASE_DIR / "data" / "raw"
RESULTS_FIGURES = BASE_DIR / "results" / "figures"
plt.rc('font', family='sans-serif')
plt.rcParams['axes.unicode_minus'] = False
sns.set_theme(style="whitegrid")
train_df = pd.read_csv(DATA_RAW / "train.csv")
print("[+] Step 3: 라이브러리 및 폰트 세팅 완료")

In [17]:
# [EDA 1] 시장 규모별 데이터 인스턴스 불균형 분석
plt.figure(figsize=(8, 5))
sns.countplot(data=train_df, x="group", order=["Large", "Medium", "Small"], palette="Set2")
plt.title("Data Instance Distribution by Market Cap Group", fontsize=14, fontweight='bold', pad=15)
plt.savefig(RESULTS_FIGURES / "eda1_market_cap_distribution_en.png", dpi=300, bbox_inches="tight")
plt.show()

In [18]:
# [EDA 2] 시장 규모별 거래량 절대값 분포 분석 (로그 스케일)
plt.figure(figsize=(10, 6))
sns.boxplot(data=train_df, x="group", y="volume", order=["Large", "Medium", "Small"], palette="Set2")
plt.yscale("log")
plt.title("Trading Volume Distribution by Group (Log Scale)", fontsize=14, fontweight='bold', pad=15)
plt.savefig(RESULTS_FIGURES / "eda2_volume_boxplot_log_en.png", dpi=300, bbox_inches="tight")
plt.show()

In [19]:
# [EDA 3] 일간 수익률 변동성 분포 및 꼬리 위험 분석
df_eda3 = train_df.sort_values(["ticker", "date"]).copy()
df_eda3["prev_close"] = df_eda3.groupby("ticker")["close"].shift(1)
df_eda3["daily_return"] = (df_eda3["close"] - df_eda3["prev_close"]) / df_eda3["prev_close"]
plt.figure(figsize=(10, 6))
sns.kdeplot(data=df_eda3, x="daily_return", hue="group", common_norm=False, fill=True, alpha=0.2, palette="Set2")
plt.xlim(-0.15, 0.15)
plt.title("Daily Return Density Plot by Market Cap Group", fontsize=14, fontweight='bold', pad=15)
plt.savefig(RESULTS_FIGURES / "eda3_daily_return_density_en.png", dpi=300, bbox_inches="tight")
plt.show()

In [20]:
# [EDA 4] 거래량 변화율과 5일 이동 변동성 관계 분석 (영문 차트 세팅)
df_eda4 = train_df.sort_values(["ticker", "date"]).copy()
df_eda4["prev_volume"] = df_eda4.groupby("ticker")["volume"].shift(1)
df_eda4["vol_chg_rate"] = (df_eda4["volume"] - df_eda4["prev_volume"]) / df_eda4["prev_volume"]
df_eda4["prev_close"] = df_eda4.groupby("ticker")["close"].shift(1)
df_eda4["daily_return"] = (df_eda4["close"] - df_eda4["prev_close"]) / df_eda4["prev_close"]
df_eda4["volatility_5d"] = df_eda4.groupby("ticker")["daily_return"].transform(lambda x: x.rolling(5).std())
df_filtered = df_eda4[(df_eda4["vol_chg_rate"] <= df_eda4["vol_chg_rate"].quantile(0.99)) & (df_eda4["volatility_5d"] <= df_eda4["volatility_5d"].quantile(0.99))]
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df_filtered, x="vol_chg_rate", y="volatility_5d", hue="group", alpha=0.4, palette="Set2")
plt.title("Relationship between Trading Volume and Price Volatility", fontsize=14, fontweight='bold', pad=15)
plt.savefig(RESULTS_FIGURES / "eda4_volume_vs_volatility_en.png", dpi=300, bbox_inches="tight")
plt.show()

In [21]:
# [EDA 5] 특정 이상 징후 종목의 캔들 패턴(OHLCV) 시각화
!pip install mplfinance -q
import mplfinance as mpf
sample_ticker = train_df[train_df["group"] == "Small"]["ticker"].iloc[0]
df_candle = train_df[train_df["ticker"] == sample_ticker].sort_values("date").copy().set_index("date")
df_candle.index = pd.to_datetime(df_candle.index)
df_candle.index.name = "Date"
mc = mpf.make_marketcolors(up='green', down='red', inherit=True)
s = mpf.make_mpf_style(base_mpf_style='charles', marketcolors=mc, gridstyle='--')
mpf.plot(df_candle.tail(30), type='candle', volume=True, style=s, figsize=(11, 7), title=f"OHLCV Candlestick Analysis - {sample_ticker}", savefig=dict(fname=str(RESULTS_FIGURES / 'eda5_candlestick_anomaly_en.png'), dpi=300, bbox_inches='tight'))

In [22]:
# [EDA 6] 주요 금융 변수 간 상관관계(Correlation) 분석 행렬
df_corr_target = df_eda4[["open", "high", "low", "close", "volume", "daily_return", "volatility_5d"]].dropna()
plt.figure(figsize=(9, 7))
sns.heatmap(df_corr_target.corr(), annot=True, cmap="coolwarm", fmt=".2f", linewidths=0.5, vmin=-1, vmax=1)
plt.title("Feature Correlation Matrix", fontsize=14, fontweight='bold', pad=15)
plt.savefig(RESULTS_FIGURES / "eda6_feature_correlation_en.png", dpi=300, bbox_inches="tight")
plt.show()

In [23]:
# [EDA 7] 데이터 내 이상 패턴(Extreme Outlier) 존재 여부 검증 분포
plt.figure(figsize=(10, 6))
sns.histplot(df_filtered["vol_chg_rate"], bins=100, kde=True, color="teal", stat="probability")
plt.title("Distribution of Volume Change Rate & Anomaly Tails", fontsize=14, fontweight='bold', pad=15)
plt.savefig(RESULTS_FIGURES / "eda7_anomaly_existence_en.png", dpi=300, bbox_inches="tight")
plt.show()